In [1]:
import sys
sys.path.insert(0, "../")   # so "src" is findable from notebooks/

from src.spark_session import get_spark
from src.data_loader import load_ratings, load_users, load_movies, join_all

spark   = get_spark()
ratings = load_ratings(spark)
ratings.printSchema()
ratings.count()


root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Timestamp: long (nullable = true)



1000209

In [2]:
users = load_users(spark)
users.printSchema()
users.count()

root
 |-- UserID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- ZipCode: string (nullable = true)



6040

In [3]:
movies = load_movies(spark)
movies.printSchema()
movies.count()

root
 |-- MovieID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)



3883

In [4]:
joins = join_all(ratings, users, movies)
joins.printSchema()
print("Row count:", joins.count())
print("Column count:", len(joins.columns))
joins.show(5)

root
 |-- MovieID: integer (nullable = true)
 |-- UserID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Timestamp: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- ZipCode: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)

Row count: 1000209
Column count: 10
+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|ZipCode|               Title|              Genres|
+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|   1193|     1|   5.0|978300760|     F|  1|        10|  48067|One Flew Over the...|               Drama|
|    661|     1|   3.0|978302109|     F|  1|        10|  48067|James and the Gia...|Animation|Childre...|
|    914|     1|   3.0|978301968|     F|  

In [5]:
from src.eda import count_unique_genres, avg_rating_age_group, top_movie_by_ratings

joined = join_all(ratings, load_users(spark), load_movies(spark))

print("Unique genres:", count_unique_genres(joined))
print("Avg rating (25-34):", avg_rating_age_group(joined))

title, count = top_movie_by_ratings(joined)
print(f"Most rated: {title} ({count} ratings)")


Unique genres: 18
Avg rating (25-34): 3.55
Most rated: American Beauty (1999) (3428 ratings)


In [6]:
import sys
sys.path.insert(0, "../")

from src.data_cleaning import run_all_checks
from src.spark_session import get_spark
from src.data_loader import load_ratings, load_users, load_movies, join_all

spark = get_spark()
ratings = load_ratings(spark)
users = load_users(spark)
movies = load_movies(spark)
joined = join_all(ratings, users, movies)

results = run_all_checks(ratings, users, movies, joined)

for rr in results:
    status = "✅ PASS" if rr["passed"] else "❌ FAIL"
    print(f"{status}  {rr['check']}: {rr['detail']}")

✅ PASS  Duplicate Ratings: 1,000,209 rows | 1,000,209 unique pairs | duplicates: 0
✅ PASS  UserID Referential Integrity: UserIDs in ratings not found in users table: 0
✅ PASS  MovieID Referential Integrity: MovieIDs in ratings not found in movies table: 0
✅ PASS  Rating Value Validity: Rows with Rating outside [1.0, 2.0, 3.0, 4.0, 5.0]: 0
✅ PASS  Age Code Validity: Rows with Age outside [1, 18, 25, 35, 45, 50, 56]: 0
✅ PASS  Occupation Code Validity: Rows with Occupation outside [0, 20]: 0
✅ PASS  Null Audit: Total nulls: 0 - all columns clean


In [7]:
from src.feature_engineering import build_features

df_featured = build_features(joined)
print(f"Columns: {len(df_featured.columns)}")
df_featured.show(5)

Columns: 27
+------+-------+------+---------+------+---+----------+-------+--------------------+--------------------+-----------+------------------+----------------+--------------------+------------+---------+------------------+-----------------+-------------------+----------------------+-------------+----------+------------+------+---------+---------+------------+
|UserID|MovieID|Rating|Timestamp|Gender|Age|Occupation|ZipCode|               Title|              Genres|high_rating|  movie_avg_rating|movie_popularity|log_movie_popularity|release_year|movie_age|   user_avg_rating|user_rating_count|   rating_deviation|user_movie_interaction|primary_genre|num_genres|is_film_noir|is_war|is_horror|is_action|   movie_era|
+------+-------+------+---------+------+---+----------+-------+--------------------+--------------------+-----------+------------------+----------------+--------------------+------------+---------+------------------+-----------------+-------------------+----------------------

In [10]:
from src.utils import processed_path
import os

# Create the folder if it doesn't exist yet
os.makedirs(processed_path(""), exist_ok=True)

# Configure Spark to avoid Hadoop permission issues on Windows
spark.conf.set("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem")
spark.conf.set("spark.hadoop.fs.file.impl.disable.cache", "true")

# Save
output_path = processed_path("featured_d2.parquet")
df_featured.write.mode("overwrite").parquet(output_path)

print(f"Saved {df_featured.count():,} rows to:")
print(f"  {output_path}")


Py4JJavaError: An error occurred while calling o421.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:735)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:270)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:286)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:978)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:660)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:700)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:672)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:699)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.j

In [ ]:
from src.eda import correlation_matrix, feature_target_summary

numeric_features = [
    "user_movie_interaction", "movie_avg_rating", "user_avg_rating",
    "rating_deviation", "movie_popularity", "log_movie_popularity",
    "movie_age", "num_genres", "high_rating"
]

# Heatmap
fig = correlation_matrix(df_featured, numeric_features)
fig.show()

# Ranked table
summary = feature_target_summary(df_featured, numeric_features[:-1])
print(summary)


                  feature  correlation  abs_corr
0  user_movie_interaction       0.4834    0.4834
1        movie_avg_rating       0.4102    0.4102
2         user_avg_rating       0.3376    0.3376
3        rating_deviation       0.3376    0.3376
4        movie_popularity       0.2156    0.2156
5    log_movie_popularity       0.2117    0.2117
6               movie_age       0.1309    0.1309
7              num_genres      -0.0039    0.0039
